# RF3: Dynamic Baselines for EA-NPS
Early stopping + random freezing vs EA-NPS on PermutedMNIST (fast decay).
Output: `dynamic_baselines.csv`

In [ ]:
import sys, subprocess, time, warnings

try:
    import avalanche
    print("Avalanche is already installed.")
except ImportError:
    print("Installing avalanche-lib...")
    res = subprocess.run(
        [sys.executable, "-m", "pip", "install",
            "avalanche-lib", "-q", "--no-warn-conflicts"],
        capture_output=True, text=True
    )
    if res.returncode != 0:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "avalanche-lib", "-q"])
    print("Installation successful.")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import OrderedDict
from avalanche.benchmarks.classic import PermutedMNIST
from avalanche.training.templates import SupervisedTemplate
from avalanche.training.plugins import SupervisedPlugin
from avalanche.training import Replay

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

### Model & Precomputed MACs

In [ ]:
def make_model():
    return nn.Sequential(OrderedDict([
        ("flatten", nn.Flatten()),
        ("fc1", nn.Linear(28*28, 256)), ("relu1", nn.ReLU()),
        ("fc2", nn.Linear(256, 128)), ("relu2", nn.ReLU()),
        ("fc3", nn.Linear(128, 10)),
    ]))

def precompute_macs():
    m = make_model()
    fwd = sum(mod.in_features * mod.out_features for mod in m.modules() if isinstance(mod, nn.Linear))
    bwd = fwd * 2
    return {
        "sgd": fwd + bwd,
        "er": fwd + bwd + fwd * 0.5,
        "freeze": fwd + bwd * 0.4,
        "es": fwd + bwd
    }

STATIC_MACS = precompute_macs()
print(f"Model MACs: {STATIC_MACS}")

### NPS Computation

In [ ]:
class FastNPS:
    def __init__(self, model, buffer, dev):
        self.model = model; self.buffer = buffer; self.device = dev
        self.criterion = nn.CrossEntropyLoss()

    def compute(self, x_new, y_new):
        if len(self.buffer) < 10:
            return 1.0
        idx = np.random.choice(len(self.buffer), min(64, len(self.buffer)), replace=False)
        old_x = torch.stack([self.buffer[i][0] for i in idx]).to(self.device).detach()
        old_y = torch.tensor([self.buffer[i][1] for i in idx], device=self.device)
        self.model.zero_grad()
        self.criterion(self.model(old_x), old_y).backward()
        g_old = torch.cat([p.grad.flatten() for p in self.model.parameters() if p.grad is not None])
        self.model.zero_grad()
        self.criterion(self.model(x_new.detach()), y_new).backward()
        g_new = torch.cat([p.grad.flatten() for p in self.model.parameters() if p.grad is not None])
        cos = nn.functional.cosine_similarity(g_old.unsqueeze(0), g_new.unsqueeze(0))
        return float(torch.clamp(1.0 - cos, 0, 1).item())

### EA-NPS Plugin (Proxy & Random Freeze)

In [ ]:
class EANPSPlugin(SupervisedPlugin):
    def __init__(self, nps_threshold=0.2, mem_size=2000, battery_decay=0.25, freeze_mode="proxy"):
        super().__init__()
        self.nps_threshold = nps_threshold; self.mem_size = mem_size
        self.battery = 1.0; self.battery_decay = battery_decay
        self.buffer = []; self.strategy_history = []
        self.current_strat = "sgd"; self.freeze_mode = freeze_mode
        self.nps_computer = None

    def before_training_exp(self, strategy, **kwargs):
        self.battery = max(0.05, self.battery - self.battery_decay)
        dl = torch.utils.data.DataLoader(strategy.experience.dataset, batch_size=64, shuffle=True, num_workers=0)
        batch = next(iter(dl))
        x_new, y_new = batch[0].to(strategy.device), batch[1].to(strategy.device)
        if self.nps_computer is None:
            self.nps_computer = FastNPS(strategy.model, self.buffer, strategy.device)
        nps = self.nps_computer.compute(x_new, y_new)
        if nps <= self.nps_threshold:
            strat = "sgd"
        elif self.battery < 0.2:
            strat = "freeze"
        else:
            strat = "er"
        self.strategy_history.append(strat)
        self.current_strat = strat
        if strat == "freeze":
            self._freeze_layers(strategy, x_new)

    def _freeze_layers(self, strategy, x_new):
        if self.freeze_mode == "proxy":
            layer_norms = {}
            for name, param in strategy.model.named_parameters():
                if param.dim() > 1:
                    ln = name.rsplit(".", 1)[0] if "." in name else name
                    layer_norms[ln] = torch.norm(param.data).item()
            sorted_layers = sorted(layer_norms.items(), key=lambda x: x[1], reverse=True)
            num = max(1, int(len(sorted_layers) * 0.1))
            top = set(n for n, _ in sorted_layers[:num])
            for name, param in strategy.model.named_parameters():
                ln = name.rsplit(".", 1)[0] if "." in name else name
                if ln in top:
                    param.requires_grad = False
        elif self.freeze_mode == "random":
            frozen = [n for n, p in strategy.model.named_parameters() if p.dim() > 1]
            num = max(1, int(len(frozen) * 0.1))
            chosen = np.random.choice(frozen, num, replace=False)
            for name, param in strategy.model.named_parameters():
                if name in chosen:
                    param.requires_grad = False
        else:
            for name, param in strategy.model.named_parameters():
                if "fc3" not in name:
                    param.requires_grad = False

    def before_training_iteration(self, strategy, **kwargs):
        if self.current_strat in ("er", "freeze") and len(self.buffer) > 0:
            ss = min(strategy.train_mb_size, len(self.buffer))
            idx = np.random.choice(len(self.buffer), ss, replace=False)
            bx = torch.stack([self.buffer[i][0] for i in idx]).to(strategy.device)
            by = torch.tensor([self.buffer[i][1] for i in idx], device=strategy.device)
            strategy.mbatch = (torch.cat([strategy.mbatch[0], bx], dim=0),
                              torch.cat([strategy.mbatch[1], by], dim=0),
                              *strategy.mbatch[2:])

    def after_training_exp(self, strategy, **kwargs):
        for p in strategy.model.parameters():
            p.requires_grad = True
        ds = strategy.experience.dataset
        idx = np.random.choice(len(ds), min(500, len(ds)), replace=False)
        for i in idx:
            img, label, *_ = ds[i]
            self.buffer.append((img.detach().cpu(), label))
        if len(self.buffer) > self.mem_size:
            self.buffer = self.buffer[-self.mem_size:]

### Early Stopping Plugin

In [ ]:
class EarlyStoppingPlugin(SupervisedPlugin):
    def __init__(self, patience=2):
        super().__init__()
        self.patience = patience; self.best_val = 0.0; self.wait = 0

    def after_training_epoch(self, strategy, **kwargs):
        model = strategy.model; model.eval()
        exp = strategy.experience
        correct, total = 0, 0
        dl = torch.utils.data.DataLoader(exp.dataset, batch_size=1024, shuffle=False, num_workers=0)
        with torch.no_grad():
            for x, y, *_ in dl:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
                total += y.size(0)
        acc = correct / total
        if acc > self.best_val:
            self.best_val = acc; self.wait = 0
        else:
            self.wait += 1
        if self.wait >= self.patience:
            strategy.stop_training = True

### Runner

In [ ]:
def run_strategy(strategy_name, seed, n_tasks=5, epochs=3, mem_size=2000, batch_size=128):
    torch.manual_seed(seed); np.random.seed(seed)
    bm = PermutedMNIST(n_experiences=n_tasks, seed=seed)
    model = make_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    t0 = time.time()
    routes = []

    if strategy_name == "ea_nps_proxy":
        plugin = EANPSPlugin(freeze_mode="proxy")
        strategy = SupervisedTemplate(model=model, optimizer=optimizer,
            criterion=nn.CrossEntropyLoss(), train_mb_size=batch_size, train_epochs=epochs,
            device=device, plugins=[plugin])
        for exp in bm.train_stream: strategy.train(exp)
        routes = plugin.strategy_history

    elif strategy_name == "ea_nps_random":
        plugin = EANPSPlugin(freeze_mode="random")
        strategy = SupervisedTemplate(model=model, optimizer=optimizer,
            criterion=nn.CrossEntropyLoss(), train_mb_size=batch_size, train_epochs=epochs,
            device=device, plugins=[plugin])
        for exp in bm.train_stream: strategy.train(exp)
        routes = plugin.strategy_history

    elif strategy_name == "early_stop":
        strategy = SupervisedTemplate(model=model, optimizer=optimizer,
            criterion=nn.CrossEntropyLoss(), train_mb_size=batch_size, train_epochs=epochs,
            device=device, plugins=[EarlyStoppingPlugin(patience=2)])
        for exp in bm.train_stream: strategy.train(exp)
        routes = ["es"] * n_tasks

    elif strategy_name == "er":
        strategy = Replay(model=model, optimizer=optimizer,
            criterion=nn.CrossEntropyLoss(), mem_size=mem_size,
            train_mb_size=batch_size, train_epochs=epochs, device=device)
        for exp in bm.train_stream: strategy.train(exp)
        routes = ["er"] * n_tasks

    else:
        raise ValueError(f"Unknown strategy: {strategy_name}")

    wall = time.time() - t0

    # Evaluate final accuracy
    model.eval()
    correct, total = 0, 0
    for exp in bm.test_stream:
        dl = torch.utils.data.DataLoader(exp.dataset, batch_size=1024, num_workers=0, pin_memory=True)
        with torch.no_grad():
            for x, y, *_ in dl:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
                total += y.size(0)
    acc = correct / total

    macs_per_route = [STATIC_MACS.get(r, STATIC_MACS["er"]) for r in routes]
    total_macs = sum(macs_per_route)
    macs_saved = round((1 - total_macs / (STATIC_MACS["er"] * n_tasks)) * 100, 1)

    return {"strategy": strategy_name, "seed": seed, "accuracy": round(acc, 4),
            "wall_time": round(wall, 1), "macs_saved_pct": macs_saved,
            "routes": "\u2192".join(routes)}

### Main: Run All Trials

In [ ]:
strategies = ["ea_nps_proxy", "ea_nps_random", "early_stop", "er"]
seeds = [42, 43, 44]
all_rows = []

print(f"Running {len(strategies)} strategies \u00d7 {len(seeds)} seeds = {len(strategies)*len(seeds)} trials")
for strat in strategies:
    for seed in seeds:
        print(f"\n--- {strat} seed={seed} ---")
        row = run_strategy(strat, seed)
        all_rows.append(row)
        print(f">> Acc={row['accuracy']:<6} Time={row['wall_time']:<5}s MACs={row['macs_saved_pct']:>5}% Routes={row['routes']}")

In [ ]:
df = pd.DataFrame(all_rows)
df.to_csv("dynamic_baselines.csv", index=False)
print("=== dynamic_baselines.csv ===")
print(df.to_string())

In [ ]:
summary = df.groupby("strategy").agg(
    acc_mean=("accuracy", "mean"), acc_std=("accuracy", "std"),
    time_mean=("wall_time", "mean"), macs_mean=("macs_saved_pct", "mean")
).round(4)
print("\n=== Summary ===")
print(summary.to_string())
print("\nDone. Copy dynamic_baselines.csv to vip_res/")